# Sanity Check for taste_vector.npy

In [8]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/claraoberle/personal-book-recommender")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.build_k_nearest_neighbours import (
    get_eligible_books,
    score_candidate_knn,
    validate_knn,
    score_to_read,
)

EMBEDDINGD_PATH = "/home/claraoberle/personal-book-recommender/data/embedding/embeddings.npy"
BOOK_ID_PATH = "/home/claraoberle/personal-book-recommender/data/embedding/book_ids.npy"
TASTE_VECTOR_PATH = "/home/claraoberle/personal-book-recommender/data/embedding/taste_vector.npy"
CLEAN_DF_PATH = "/home/claraoberle/personal-book-recommender/data/embedding/embedding_join_clean_df.csv"

In [9]:
# Load data
embeddings = np.load(EMBEDDINGD_PATH, allow_pickle=True)
book_ids = np.load(BOOK_ID_PATH, allow_pickle=True)
taste_vector = np.load(TASTE_VECTOR_PATH, allow_pickle=True)

clean_df = pd.read_csv(CLEAN_DF_PATH)

clean_df["Book Id"] = clean_df["Book Id"].astype(str)

In [5]:
# Cosine similarity between taste vector and every book
similarities = embeddings @ taste_vector

# Create dataframe with results
similarity_df = pd.DataFrame({"Book Id": book_ids.astype(str), "similarity": similarities})

# Add book information
similarity_df = similarity_df.merge(clean_df[["Book Id", "Title", "My Rating", "Exclusive Shelf"]], on="Book Id", how="left")

## Top 10 books sorted by similarity

In [6]:
# Sort by similarity
top_10 = similarity_df.sort_values("similarity", ascending=False).head(10)

print(top_10[["Title", "similarity", "My Rating", "Exclusive Shelf"]].to_string(index=False))

                                                                  Title  similarity  My Rating   Exclusive Shelf
              Harry Potter and the Half-Blood Prince (Harry Potter, #6)    0.475732        5.0              read
           Harry Potter and the Order of the Phoenix (Harry Potter, #5)    0.469269        5.0              read
                Harry Potter and the Deathly Hallows (Harry Potter, #7)    0.412358        5.0              read
            Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)    0.409185        5.0              read
                                 Fire & Blood (A Targaryen History, #1)    0.405359        0.0           to-read
                  La comunidad del anillo (El señor de los anillos, #1)    0.384264        0.0 currently-reading
                 Harry Potter and the Goblet of Fire (Harry Potter, #4)    0.339458        5.0              read
             Harry Potter and the Chamber of Secrets (Harry Potter, #2)    0.319459        5.0  

In [7]:
print(f"Mean similarity: {similarities.mean():.3f}")
print(f"Min similarity: {similarities.min():.3f}")
print(f"Max similarity: {similarities.max():.3f}")

Mean similarity: -0.019
Min similarity: -0.388
Max similarity: 0.476


The similarity check produced encouraging results. The top-ranked books are dominated by highly rated Harry Potter books, with Fire & Blood ranking fifth with a similarity of 0.406 despite being on the to-read shelf and therefore being excluded from the construction of the taste vector. La comunidad del anillo also ranks highly despite being currently read and unrated, meaning it was likewise excluded from the taste vector. This suggests that the taste vector is capturing meaningful patterns in my reading preferences rather than simply reproducing the books used to construct it. However, the top 10 are heavily dominated by the Harry Potter series, which highlights a potential limitation of the approach: because the taste vector is an average of the weighted book embeddings, several books from the same tightly clustered series can reinforce the same region of the embedding space. As a result, the taste vector may be biased towards genres or series that I have read many books from, rather than representing all genres that I rate highly equally. To investigate this further, I will examine the similarity scores of the highest-ranked books and also inspect the lowest similarities to see how the taste vector distinguishes books that are less aligned with my preferences.

## High rated books

In [ ]:
high_rated = similarity_df[similarity_df["My Rating"] >= 4].sort_values("similarity", ascending=False)

print(high_rated[["Title", "similarity", "My Rating"]].head(10).to_string(index=False))

                                                       Title  similarity  My Rating
   Harry Potter and the Half-Blood Prince (Harry Potter, #6)    0.475732        5.0
Harry Potter and the Order of the Phoenix (Harry Potter, #5)    0.469269        5.0
     Harry Potter and the Deathly Hallows (Harry Potter, #7)    0.412358        5.0
 Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)    0.409185        5.0
      Harry Potter and the Goblet of Fire (Harry Potter, #4)    0.339458        5.0
  Harry Potter and the Chamber of Secrets (Harry Potter, #2)    0.319459        5.0
                                 Archenemies (Renegades, #2)    0.280174        5.0
 Harry Potter and the Philosopher's Stone (Harry Potter, #1)    0.264268        5.0
           A Dance with Dragons (A Song of Ice and Fire, #5)    0.234550        5.0
                           Mockingjay (The Hunger Games, #3)    0.229457        5.0


Bottom of high rated books:

In [9]:
high_rated = similarity_df[similarity_df["My Rating"] >= 4].sort_values("similarity", ascending=True)

print(high_rated[["Title", "similarity", "My Rating"]].head(10).to_string(index=False))

                                                                    Title  similarity  My Rating
                                                        Love on the Brain   -0.278132        4.0
                                              The Roommate (Shameless #1)   -0.241500        4.0
            P.S. I Still Love You (To All the Boys I've Loved Before, #2)   -0.222047        4.0
                                        El amor en los tiempos del cólera   -0.207248        4.0
                                                          The Nightingale   -0.193039        4.0
                          Get a Life, Chloe Brown (The Brown Sisters, #1)   -0.190286        4.0
                                                      Pride and Prejudice   -0.163815        4.0
                                          Six of Crows (Six of Crows, #1)   -0.161110        5.0
                                                  Una segunda oportunidad   -0.151830        4.0
To All the Boys I've Loved Bef

The bottom of the list of highly rated books reveals an important limitation of the single taste vector. Every book in this group was rated 4 or 5 stars, yet all of them have negative similarity with the taste vector. The pattern is also meaningful: many are contemporary romance or literary/historical fiction, such as Love on the Brain, The Roommate, Pride and Prejudice, and The Nightingale. Even Six of Crows, a fantasy title rated 5 stars, has a negative similarity, suggesting that the embedding space distinguishes between different types of fantasy rather than treating them as one broad preference.

This is explained by the fact that the mean similarity across the library is approximately -0.019, which is essentially zero. Averaging embeddings from books belonging to different preference clusters can cause their directions to partially cancel each other out. In this case, the taste vector appears to be dominated by the tightly clustered fantasy books, while other genres that I also rate highly point in different directions and therefore become poorly represented by the single average vector. This is a useful finding rather than simply a failure of the approach: it suggests that my preferences are better represented by multiple clusters or separate taste vectors than by a single averaged vector, which supports experimenting with alternative representations of preferences in the next stage of the recommender.

## K-nearest neighbour

In [13]:
def inspect_book_knn(book_id, clean_df, eligible_vectors, eligible_ratings, eligible_ids, mean_rating, embeddings, book_ids, k=5):
    """Inspect the k nearest eligible neighbours for a given book."""

    book_id = str(book_id)

    # Find the book in the embedding array
    embedding_indices = np.where(book_ids.astype(str) == book_id)[0]

    if len(embedding_indices) == 0:
        raise ValueError(f"Book Id {book_id} not found in book_ids.")

    book_vector = embeddings[embedding_indices[0]]

    # Find the book's information
    book_info = clean_df[clean_df["Book Id"] == book_id]

    if book_info.empty:
        raise ValueError(f"Book Id {book_id} not found in clean_df.")

    # Calculate similarity to all eligible books
    similarities = eligible_vectors @ book_vector

    # Exclude the book itself
    mask = eligible_ids != book_id

    similarities = similarities[mask]
    ratings = eligible_ratings[mask]
    ids = eligible_ids[mask]

    # Get k most similar books
    top_k_idx = similarities.argsort()[::-1][:k]

    # Create result dataframe
    neighbors = pd.DataFrame({
        "Book Id": ids[top_k_idx],
        "similarity": similarities[top_k_idx],
        "My Rating": ratings[top_k_idx]
    })

    # Add titles
    neighbors = neighbors.merge(
        clean_df[["Book Id", "Title"]],
        on="Book Id",
        how="left"
    )

    # Calculate predicted rating
    predicted_deviation = (neighbors["My Rating"] - mean_rating).mean()

    predicted_rating = mean_rating + predicted_deviation

    print(f"Book: {book_info.iloc[0]['Title']}")
    print(f"Book Id: {book_id}")
    print(f"Actual rating: {book_info.iloc[0]['My Rating']}")
    print(f"Mean rating: {mean_rating:.3f}")
    print(f"Predicted rating: {predicted_rating:.3f}")
    print("\nNearest neighbours:")

    print(neighbors[["Title", "similarity", "My Rating"]].to_string(index=False))

    return neighbors

In [14]:
eligible_vectors, eligible_ratings, eligible_ids, mean_rating = get_eligible_books(clean_df, embeddings, book_ids)

In [15]:
harry_potter_6_id = "1"
archenemies_id = "35425827"
six_of_crows_id = "23437156"
# to all the boys i have loved before
tatbihlb_id = "15749186"
pride_prejudice_id = "1885"
the_nightingale_id = "21853621"

### Trying different (random) books

In [16]:
neighbors = inspect_book_knn(harry_potter_6_id, clean_df, eligible_vectors, eligible_ratings, 
                             eligible_ids, mean_rating, embeddings, book_ids, k=5)


Book: Harry Potter and the Half-Blood Prince (Harry Potter, #6)
Book Id: 1
Actual rating: 5.0
Mean rating: 3.897
Predicted rating: 4.600

Nearest neighbours:
                                                                  Title  similarity  My Rating
           Harry Potter and the Order of the Phoenix (Harry Potter, #5)    0.690136        5.0
                Harry Potter and the Deathly Hallows (Harry Potter, #7)    0.664030        5.0
            Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)    0.633990        5.0
Harry Potter and the Cursed Child: Parts One and Two (Harry Potter, #8)    0.578720        3.0
                 Harry Potter and the Goblet of Fire (Harry Potter, #4)    0.563999        5.0


In [17]:
neighbors = inspect_book_knn(archenemies_id, clean_df, eligible_vectors, eligible_ratings, 
                             eligible_ids, mean_rating, embeddings, book_ids, k=5)


Book: Archenemies (Renegades, #2)
Book Id: 35425827
Actual rating: 5.0
Mean rating: 3.897
Predicted rating: 4.200

Nearest neighbours:
                                            Title  similarity  My Rating
                        Supernova (Renegades, #3)    0.681522        5.0
                        Renegades (Renegades, #1)    0.592739        5.0
Siege and Storm (The Shadow and Bone Trilogy, #2)    0.540911        3.0
            Shadow and Bone (Shadow and Bone, #1)    0.537458        4.0
                             All You Need Is Kill    0.504412        4.0


In [18]:
neighbors = inspect_book_knn(six_of_crows_id, clean_df, eligible_vectors, eligible_ratings, 
                             eligible_ids, mean_rating, embeddings, book_ids, k=5)


Book: Six of Crows (Six of Crows, #1)
Book Id: 23437156
Actual rating: 5.0
Mean rating: 3.897
Predicted rating: 3.400

Nearest neighbours:
                                         Title  similarity  My Rating
         Ruin and Rising (Shadow and Bone, #3)    0.980788        4.0
                                      The Help    0.606038        3.0
Todo lo que nunca fuimos (Deja que ocurra, #1)    0.599765        3.0
                           Pride and Prejudice    0.587568        4.0
                              Heist (Darks #1)    0.573541        3.0


In [19]:
neighbors = inspect_book_knn(tatbihlb_id, clean_df, eligible_vectors, eligible_ratings, 
                             eligible_ids, mean_rating, embeddings, book_ids, k=5)


Book: To All the Boys I've Loved Before (To All the Boys I've Loved Before, #1)
Book Id: 15749186
Actual rating: 4.0
Mean rating: 3.897
Predicted rating: 3.800

Nearest neighbours:
                                                                Title  similarity  My Rating
Always and Forever, Lara Jean (To All the Boys I've Loved Before, #3)    0.883541        3.0
            Dune (edición especial película) (Las crónicas de Dune 1)    0.741512        5.0
        P.S. I Still Love You (To All the Boys I've Loved Before, #2)    0.721945        4.0
                                                Caraval (Caraval, #1)    0.663960        2.0
                                                   The Secret History    0.621013        5.0


In [ ]:
neighbors = inspect_book_knn(pride_prejudice_id, clean_df, eligible_vectors, eligible_ratings, 
                             eligible_ids, mean_rating, embeddings, book_ids, k=5)

Book: Pride and Prejudice
Book Id: 1885
Actual rating: 4.0
Mean rating: 3.897
Predicted rating: 3.200

Nearest neighbours:
                                          Title  similarity  My Rating
 Todo lo que nunca fuimos (Deja que ocurra, #1)    0.663788        3.0
                     Margo's Got Money Troubles    0.622791        3.0
Call Me By Your Name (Call Me By Your Name, #1)    0.595939        3.0
                            Love, Theoretically    0.595829        3.0
          Ruin and Rising (Shadow and Bone, #3)    0.595540        4.0


In [21]:
neighbors = inspect_book_knn(the_nightingale_id, clean_df, eligible_vectors, eligible_ratings, 
                             eligible_ids, mean_rating, embeddings, book_ids, k=5)


Book: The Nightingale
Book Id: 21853621
Actual rating: 4.0
Mean rating: 3.897
Predicted rating: 3.200

Nearest neighbours:
                                     Title  similarity  My Rating
                          Mientras vivimos    0.663526        3.0
         Me Before You (Me Before You, #1)    0.655662        3.0
Los pilares de la tierra (Kingsbridge, #1)    0.612283        4.0
                       The Love Hypothesis    0.608644        3.0
                Margo's Got Money Troubles    0.558083        3.0


### All read books